# Tool Calling

**Tool Calling** เป็นเทคนิคหนึ่งที่มีความสำคัญอย่างมากในปัจจุบัน เนื่องจากเราพบว่าในความเป็นจริงแล้ว AI ไม่ได้เก่งไปเสียทุกเรื่องอย่างที่คาดหวัง แทนที่จะพยายามสร้าง AI ให้ทำทุกอย่างได้ด้วยตัวเอง แนวทางใหม่ๆจึงเปลี่ยนวิธีเป็นการสอนให้ AI _"เรียนรู้วิธีการใช้เครื่องมือ (Tools)"_ นั้นคือ เราจะให้ AI ทำหน้าที่เน้นไปในการคิด วางแผน และตัดสินใจ แล้วมอบให้ Tools ต่างๆ จัดการกับงานเฉพาะทางแทน


ใน Lab นี้จะยกตัวอย่าง 2 use cases ในการใช้ Tool calling 
   1. ระบบแก้โจทย์คณิตศาสตร์
   2. ระบบตอบคำถามจากฐานข้อมูล

#### 1. ระบบแก้โจทย์คณิตศาสตร์
ภารกิจนี้จะให้ AI ทำหน้าที่เป็นผู้วางแผนแก้โจทย์ปัญหาจากข้อความง่ายๆ ระดับประถม เช่น _ฉันมีแอปเปิล 10 ผล น้องกินไป 2 ผล เหลือแอปเปิลกี่ผล?_

โดยใช้เครื่องคิดเลขง่ายๆที่เราสร้างขึ้นมา

**เครื่องมือที่ใช้**: `calculator(A, B, operator)`

Input: ตัวเลข A, ตัวเลข B และตัวดำเนินการ (Operator) เช่น +, -, *, /, %

Output: ผลลัพท์ A ?? B


In [6]:
# ต้องใส่ Type Hints และ Docstring ให้ชัดเจน
def calculator(A: float, B: float, operator: str) -> float | str:
    """
    เครื่องคิดเลขสำหรับคำนวณผลลัพธ์ทางคณิตศาสตร์จากตัวเลข 2 ตัว
    
    Args:
        A (float): ตัวเลขตัวแรก
        B (float): ตัวเลขตัวที่สอง
        operator (str): ตัวดำเนินการทางคณิตศาสตร์ เลือกจาก +, -, *, /, %
    """
    
    if operator == '+': 
        return A + B
    elif operator == '-': 
        return A - B
    elif operator == '*': 
        return A * B
    elif operator == '/': 
        return A / B if B != 0 else "Error: Division by zero"
    elif operator == '%': 
        return A % B
    else: 
        return "Error: Invalid operator"

In [7]:
calculator(100, 5, "*")

500

In [8]:
calculator(100, 825, "+")

925

In [20]:
import ollama

def solve_it(prompt, model_name = "scb10x/typhoon2.5-qwen3-4b"):
    messages = [
        {'role': 'system', 'content': 'คุณคือผู้ช่วยแก้โจทย์ปัญหาคณิตศาสตร์ระดับประถม จงวิเคราะห์โจทย์และใช้เครื่องมือ calculator ในการหาคำตอบให้ถูกต้องเสมอ'},
        {'role': 'user', 'content': f"หาคำตอบจากโจทย์ต่อไปนี้: {prompt}"}
    ]
    
    response = ollama.chat(
        model=model_name,
        messages=messages,
        tools=[calculator]
    )
    
    messages.append(response['message'])
    
    # ตรวจสอบการเรียกใช้ Tool
    while response['message'].get('tool_calls'):
        for tool_call in response['message']['tool_calls']:
            if tool_call['function']['name'] == 'calculator':
                args = tool_call['function']['arguments']
                
                
                result = calculator(**args)
                print(f"Tool: calculator({args}) => ผลลัพธ์: {result}")
                
                # ส่งผลลัพธ์กลับ
                messages.append({
                    'role': 'tool',
                    'content': str(result),
                    'name': 'calculator'
                })
        
        # คุยรอบสุดท้ายเพื่อสรุปคำตอบ
        response = ollama.chat(
            model=model_name,
            messages=messages,
            tools=[calculator]
        )
        
        messages.append(response['message'])
        
    print(f"Result:\n{response['message']['content']}")

# while True:
#     prompt = input('Enter your message: ')
#     if prompt.lower() == 'q':
#         break
#     else:

response = solve_it("ฉันมีแอปเปิล 10 ผล น้องกินไป 2 ผล เหลือแอปเปิลกี่ผล?")

Tool: calculator({'A': 10, 'B': 2, 'operator': '-'}) => ผลลัพธ์: 8
Result:
ฉันมีแอปเปิล 10 ผล น้องกินไป 2 ผล เหลือแอปเปิล 8 ผล


In [26]:
response = solve_it("ฉันมีแอปเปิล 19023820 ผล หนอนกินไป 29829 ผล ในส่วนที่เหลือโดนนกแย่งกินไปอีก 22350 ผล ต่อมาแอปเปิ้ลเพิ่มจำนวนขึ้น 2440 ลูก สุดท้ายเหลือแอปเปิลกี่ผล?")

Tool: calculator({'A': 19023820, 'B': 29829, 'operator': '-'}) => ผลลัพธ์: 18993991
Tool: calculator({'A': 18993991, 'B': 22350, 'operator': '-'}) => ผลลัพธ์: 18971641
Tool: calculator({'A': 18971641, 'B': 2440, 'operator': '+'}) => ผลลัพธ์: 18974081
Result:
สุดท้ายเหลือแอปเปิ้ล **18,974,081** ผล


#### 2. ระบบตอบคำถามจากฐานข้อมูล
ภารกิจนี้จะให้ AI ทำหน้าที่เป็นผู้วางแผนสร้าง SQL query เพื่อใช้ตอบคำถามโดยอ้างอิงข้อมูลจาก DB (ไฟล์ SQLite: `Example/example.db`)

**เครื่องมือที่ใช้ #1**: `execute_query(query)`

Input: SQL query

Output: ผลลัพท์จาก DB


**เครื่องมือที่ใช้ #2**: `sql_validation(query)`

Input: SQL query

Output: True ถ้า query เป็น query ที่ฟอร์เมตถูกต้อง



![](./Examples/er.png)

In [34]:
sqlSchema = '''
CREATE TABLE "artists"
(
    [ArtistId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "albums"
(
    [AlbumId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Title] NVARCHAR(160)  NOT NULL,
    [ArtistId] INTEGER  NOT NULL,
    FOREIGN KEY ([ArtistId]) REFERENCES "artists" ([ArtistId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION
);

CREATE TABLE "genres"
(
    [GenreId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "media_types"
(
    [MediaTypeId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(120)
);

CREATE TABLE "tracks"
(
    [TrackId] INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL,
    [Name] NVARCHAR(200)  NOT NULL,
    [AlbumId] INTEGER,
    [MediaTypeId] INTEGER  NOT NULL,
    [GenreId] INTEGER,
    [Composer] NVARCHAR(220),
    [Milliseconds] INTEGER  NOT NULL,
    [Bytes] INTEGER,
    [UnitPrice] NUMERIC(10,2)  NOT NULL,
    FOREIGN KEY ([AlbumId]) REFERENCES "albums" ([AlbumId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION,
    FOREIGN KEY ([GenreId]) REFERENCES "genres" ([GenreId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION,
    FOREIGN KEY ([MediaTypeId]) REFERENCES "media_types" ([MediaTypeId]) 
        ON DELETE NO ACTION ON UPDATE NO ACTION
);
'''

In [35]:
import sqlite3
import os

def execute_query(query):
    """
    Connects to a SQLite database, executes a given query, and returns the results.

    Args:
        query (str): The SQL query string to execute.

    Returns:
        list: A list of tuples, where each tuple represents a row from the results.
              Returns None if an error occurs.
    """
    connection = None
    results = None
    
    try:
        db_file = "./Examples/example.db"
        connection = sqlite3.connect(db_file)
        cursor = connection.cursor()
        
        print(f"Executing query:\n{query}\n")
        cursor.execute(query)
        results = cursor.fetchall()
        
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
        
    finally:
        if connection:
            connection.close()
            
    return results

In [36]:
execute_query("SELECT * FROM albums LIMIT 5")

Executing query:
SELECT * FROM albums LIMIT 5



[(1, 'For Those About To Rock We Salute You', 1),
 (2, 'Balls to the Wall', 2),
 (3, 'Restless and Wild', 2),
 (4, 'Let There Be Rock', 1),
 (5, 'Big Ones', 3)]

In [43]:
def sql_validation(query: str) -> bool:
    """
    Validates if the given SQL query is syntactically correct without actually fetching or modifying data.
    
    Args:
        query (str): The SQL query string to validate.
    """
    try:
        db_file = "./Examples/example.db"
        conn = sqlite3.connect(db_file)
        cursor = conn.cursor()
        cursor.execute(f"EXPLAIN {query}")
        conn.close()
        return True
    except sqlite3.Error as e:
        return False

In [45]:
sql_validation("SELECT * FROM albums LIMIT 5")

True

In [47]:
sql_validation("SELECT A, B, C FROM albums")

False

In [48]:
sql_validation("SELECT * FROM album")

False

In [1]:
import ollama

def answer_it(prompt, model_name = "scb10x/typhoon2.5-qwen3-4b"):
    messages = [
        {
            'role': 'system', 
            'content': (
                'คุณคือผู้เชี่ยวชาญด้านฐานข้อมูล SQLite '
                f'โดยอ้างอิงตามโครงสร้างฐานข้อมูล ดังต่อไปนี้ \n\n{sqlSchema}\n\n'
                'จงตอบคำถามของผู้ใช้โดยการดึงข้อมูลจาก Database  คุณสามารถใช้ execute_query เพื่อรันคำสั่งและเอาผลลัพธ์มาสรุปตอบ'
            )
        },
        {'role': 'user', 'content': f"หาคำตอบของคำถามต่อไปนี้: {prompt}"}
    ]

    response = ollama.chat(
        model=model_name,
        messages=messages,
        tools=[execute_query]
    )
    
    messages.append(response['message'])
    
    while response['message'].get('tool_calls'):
        for tool_call in response['message']['tool_calls']:
            args = tool_call['function']['arguments']
        
            if tool_call['function']['name'] == "execute_query":
                if sql_validation(**args):
                    result = execute_query(**args)
                    print(f"Tool: execute_query({args})")
                    print(f"Output: {result}")
                    print()
                else:
                    result = "SQL Failed; Please try again!"
                    print(f"Tool: execute_query({args})")
                    print(f"Output: FAILED")
                    print()
            
            messages.append({
                'role': 'tool',
                'content': str(result),
                'name': tool_call['function']['name']
            })

        
        print(messages)
        response = ollama.chat(
            model=model_name,
            messages=messages,
            tools=[execute_query]
        )
        
        messages.append(response['message'])
        
    print(f"Result:\n{response['message']['content']}")

response = answer_it("ใครเป็นนักร้องที่มีอัลบั้มเยอะที่สุด?")

NameError: name 'sqlSchema' is not defined